# 60 — Prepare the Pinatubo BOB source tree

Run this once before 61–69, and again only when new archives arrive. This is the **only BOB notebook that modifies LEGACY**: extraction creates files beside each ZIP, then moves a fully handled ZIP into `LEGACY/bob_zipfiles_obsolete/<same relative path>`. No existing file is overwritten. Preview is the default.

An existing member is accepted only when its bytes match the archive. Conflicting, damaged, unsafe or unsupported archives remain active for review; successfully recovered members are retained. This avoids hiding unresolved archives in the obsolete tree. Nested ZIPs are processed in successive passes. A copy is hash-verified before the source ZIP is removed. Keep a backup of LEGACY before applying; moving is deliberate, not deletion of the only archive copy.


## Review of the supplied saved execution

The supplied saved run was a preview with zero active ZIP archives. This confirms there was nothing to extract in that tree at that time; it does not verify archives already moved to bob_zipfiles_obsolete.

This describes the supplied version, not a new execution. Revised code cells have been cleared so stale results cannot be mistaken for current outputs. The original executed notebook is backed up.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, re
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
# Support launching Jupyter from either the repository or 10_conversion.
HERE = Path.cwd().resolve()
CONVERSION = HERE if (HERE / 'pinatubo_steps').is_dir() else HERE / '10_conversion'
if not (CONVERSION / 'pinatubo_steps').is_dir():
    raise RuntimeError('Launch Jupyter from the repository root or 10_conversion.')
if str(CONVERSION) not in sys.path: sys.path.insert(0, str(CONVERSION))
from pinatubo_steps.config import pipeline_paths, load_conversion_config
CONFIG_FILE = os.environ.get('PINATUBO_CONVERSION_CONFIG')  # None selects shared default
PATHS = pipeline_paths(config_file=CONFIG_FILE)
CONFIG = load_conversion_config(CONFIG_FILE)
BOB_CONFIG = CONFIG.get('legacy_bob', {})
BOB_ROOT = Path(PATHS['LEGACY_TOP']) / 'BOB'
BOB_OUTPUT = Path(PATHS['FAIR_TOP']) / 'BOB'
NOTEBOOK_NAME = '60_prepare_bob_archives.ipynb'
OVERWRITE = False  # Set True only to deliberately replace this notebook's derived products.
print('Inputs:', BOB_ROOT, '\nOutputs:', BOB_OUTPUT)


## Safe extraction definitions

Paths, symbolic links, duplicate/case-colliding names, file size and CRC are checked. The system `unzip` handles older compression methods. Extraction uses exclusive file creation. Reports distinguish identical existing files from clashes. The loop is bounded to avoid pathological nesting; remaining ZIPs are always listed. A preview checks visible ZIPs only; nested ZIPs will be discovered during extraction.


In [ ]:
import zipfile, stat, shutil, subprocess, zlib
from pathlib import PurePosixPath

def safe_target(root, entry):
    name = entry.filename.replace('\\','/')
    parts = PurePosixPath(name)
    if not name or parts.is_absolute() or '..' in parts.parts or ':' in name or '\x00' in name:
        raise ValueError(f'Unsafe ZIP member: {name!r}')
    if stat.S_ISLNK(entry.external_attr >> 16): raise ValueError('ZIP symlink refused')
    target = root.joinpath(*parts.parts)
    if any(p.is_symlink() for p in [target,*target.parents]):
        raise ValueError(f'Destination symlink refused: {target}')
    return target

def prepare_archives(source, obsolete, *, apply=False, max_passes=20):
    source,obsolete=Path(source).resolve(),Path(obsolete).resolve()
    if not source.is_dir(): raise FileNotFoundError(source)
    if source == obsolete or source in obsolete.parents or obsolete in source.parents:
        raise ValueError('Active and obsolete trees must not overlap')
    unzip=shutil.which('unzip')
    if not unzip: raise RuntimeError('System unzip required for historical ZIP compression')
    report=[]; attempted=set()
    for pass_number in range(max_passes):
        archives=sorted(p for p in source.rglob('*') if p.suffix.lower()=='.zip' and p.is_file() and p not in attempted)
        if not archives: break
        for archive in archives:
            attempted.add(archive)
            row=dict(archive=str(archive), pass_number=pass_number+1, members=[], status='preview')
            report.append(row)
            try:
                if any(p.is_symlink() for p in [archive,*archive.parents]): raise ValueError('Source symlink refused')
                digest=hashlib.sha256(archive.read_bytes()).hexdigest()
                row['sha256']=digest
                dest=obsolete/archive.relative_to(source);row['obsolete_path']=str(dest)
                if dest.exists(): raise FileExistsError('Obsolete archive destination already exists; original retained')
                if any(p.is_symlink() for p in [dest,*dest.parents]): raise ValueError('Obsolete path symlink refused')
                with zipfile.ZipFile(archive) as z:
                    # Preflight all names before creating any files. Reject case-fold collisions
                    # even on a case-sensitive disk, because the archive may be used on a Mac.
                    seen=set()
                    for entry in z.infolist():
                        target=safe_target(archive.parent,entry)
                        key=str(target).casefold()
                        if key in seen: raise ValueError('Duplicate/case-colliding ZIP member paths')
                        seen.add(key)
                    for entry in z.infolist():
                        target=safe_target(archive.parent,entry)
                        item=dict(member=entry.filename,destination=str(target));row['members'].append(item)
                        if entry.is_dir():
                            if target.exists() and not target.is_dir(): item['status']='clash';continue
                            if apply: target.mkdir(parents=True,exist_ok=True)
                            item['status']='directory';continue
                        # unzip supports historical compression methods not supported by Python.
                        result=subprocess.run([unzip,'-p',str(archive),entry.filename],capture_output=True,timeout=60)
                        payload=result.stdout
                        if result.returncode or len(payload)!=entry.file_size or zlib.crc32(payload)&0xffffffff != entry.CRC:
                            item.update(status='damaged',details=result.stderr.decode(errors='replace'));continue
                        item['sha256']=hashlib.sha256(payload).hexdigest()
                        if target.exists():
                            item['status']='identical_existing' if target.is_file() and hashlib.sha256(target.read_bytes()).hexdigest()==item['sha256'] else 'clash'
                        else:
                            # Detect alternate-case names on case-sensitive filesystems too.
                            if target.parent.exists() and any(p.name.casefold()==target.name.casefold() for p in target.parent.iterdir()):
                                item['status']='clash';continue
                            if apply:
                                target.parent.mkdir(parents=True,exist_ok=True)
                                with target.open('xb') as f: f.write(payload)
                            item['status']='extracted' if apply else 'would_extract'
                unresolved=any(m['status'] in ('clash','damaged') for m in row['members'])
                if unresolved: row['status']='retained_for_review'
                elif apply:
                    if hashlib.sha256(archive.read_bytes()).hexdigest()!=digest: raise RuntimeError('ZIP changed during extraction')
                    dest.parent.mkdir(parents=True,exist_ok=True)
                    with archive.open('rb') as src,dest.open('xb') as dst: shutil.copyfileobj(src,dst)
                    if hashlib.sha256(dest.read_bytes()).hexdigest()!=digest: raise RuntimeError('Archive copy verification failed; original retained')
                    archive.unlink();row['status']='moved_verified'
                else: row['status']='ready_to_extract_and_move'
            except Exception as exc: row.update(status='failed_retained',details=str(exc))
        if not apply: break  # Nested archives become visible only after an actual pass.
    remaining=[str(p) for p in source.rglob('*') if p.is_file() and p.suffix.lower()=='.zip']
    return report,remaining


In [ ]:
APPLY = False  # Review the preview, then set True to extract and move archives.
report, remaining = prepare_archives(BOB_ROOT, BOB_ROOT.parent/'bob_zipfiles_obsolete', apply=APPLY)
report_dir=BOB_OUTPUT/'archive_preparation'
report_dir.mkdir(parents=True,exist_ok=True)
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
report_path=report_dir/f'{stamp}_{"applied" if APPLY else "preview"}.json'
report_path.write_text(json.dumps(dict(applied=APPLY,archives=report,remaining=remaining),indent=2)+'\n')
display(pd.DataFrame([{k:v for k,v in row.items() if k!='members'} for row in report]))
print('Report:',report_path,'\nRemaining ZIPs:',len(remaining))
for name in remaining: print(name)


## Inventory and review

A successful rerun has no active ZIPs. Unresolved ZIPs block the relevant downstream notebook. Do not delete them to bypass that check: repair/recover the archive or explicitly select an audited input subfolder. Extraction does not deduplicate observations copied into different source folders. That is a separate scientific/provenance decision.


In [ ]:
# Inventory names/sizes without forcing cloud downloads of every source file.
inventory=[]
known={'RSAM','SSAM','DATA','TILT','RAIN','MUDFLOW','COSPEC','COUNTS','RAD_FREQ'}
for path in sorted(BOB_ROOT.rglob('*')):
    if not path.is_file():continue
    relative=path.relative_to(BOB_ROOT)
    inventory.append(dict(relative_path=str(relative),bytes=path.stat().st_size,
        top_folder=relative.parts[0] if len(relative.parts)>1 else '.',extension=path.suffix.upper(),
        reader_scope='known_product_tree' if relative.parts[0].upper() in known else 'review_other_source'))
frame=pd.DataFrame(inventory,columns=['relative_path','bytes','top_folder','extension','reader_scope'])
frame.to_csv(report_dir/f'{stamp}_source_inventory.csv',index=False)
display(frame.groupby(['top_folder','extension']).size().rename('files').reset_index())
display(frame.loc[frame.reader_scope.eq('review_other_source')])
obsolete=BOB_ROOT.parent/'bob_zipfiles_obsolete'
print('Previously moved ZIPs:',sum(p.suffix.lower()=='.zip' for p in obsolete.rglob('*') if p.is_file()) if obsolete.exists() else 0)
print('Zero active ZIPs does not independently verify previously moved/damaged archives.')
